# BasketIQ — Step 5: Market Basket Analysis

**Goal of this notebook:** find products that are frequently bought **together** in the same order, using **association rule mining** (the Apriori algorithm).

This is the classic technique behind "customers who bought X also bought Y" recommendations, and behind decisions like which products to bundle, discount together, or place near each other in a store/website.

**Note:** a new library is needed for this step — `mlxtend` has been added to `requirements.txt`. Run `pip install -r requirements.txt` again before starting.

## 1. Import libraries and load the cleaned data

In [17]:
import pandas as pd
from pathlib import Path
from mlxtend.frequent_patterns import apriori, association_rules

pd.set_option('display.max_columns', None)

In [18]:
PROCESSED_DATA_PATH = Path('..') / 'data' / 'processed' / 'online_retail_clean.csv'

df = pd.read_csv(PROCESSED_DATA_PATH)
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


## 2. Remove non-product line items

The dataset includes a few `Description` values that aren't real products — they're accounting/operational line items that happen to be recorded the same way as a product row (e.g. `POSTAGE` for shipping fees). If we leave these in, they show up in almost every invoice and pollute the results with meaningless "rules" like *"customers who buy X also pay for postage"* — true, but not useful, and it crowds out genuinely interesting product relationships.

We filter these out before building the basket.

In [19]:
# Known non-product entries in this dataset (shipping, fees, manual adjustments, etc.)
NON_PRODUCT_DESCRIPTIONS = [
    'POSTAGE',
    'DOTCOM POSTAGE',
    'CARRIAGE',
    'Manual',
    'BANK CHARGES',
    'AMAZON FEE',
    'SAMPLES',
    'ADJUST BAD DEBT',
]

rows_before = len(df)
df = df[~df['Description'].isin(NON_PRODUCT_DESCRIPTIONS)]
rows_after = len(df)

print(f'Removed {rows_before - rows_after} non-product rows ({rows_before} -> {rows_after})')

Removed 1527 non-product rows (392692 -> 391165)


## 3. Why we focus on a single country

Market basket analysis needs to build a huge table: **one row per invoice, one column per product**. With thousands of unique products and hundreds of thousands of invoices across every country mixed together, that table becomes extremely large and slow to process, and mixes shopping patterns from very different markets.

A standard approach (and the one we'll use here) is to run the analysis **per country**, starting with one market. We'll use **France**, a common and reasonably sized choice for this dataset, but you can change `COUNTRY` below to any market you're curious about.

In [20]:
# See which countries have enough transactions to make this analysis meaningful
df['Country'].value_counts().head(15)

Country
United Kingdom     348887
Germany              8643
France               8019
EIRE                 7126
Spain                2417
Netherlands          2322
Belgium              1935
Switzerland          1810
Portugal             1416
Australia            1180
Norway               1048
Channel Islands       743
Italy                 741
Finland               647
Cyprus                601
Name: count, dtype: int64

In [21]:
COUNTRY = 'France'  # <-- change this to analyze a different market

df_country = df[df['Country'] == COUNTRY]
print(f'Transactions for {COUNTRY}: {len(df_country)}')
print(f'Unique invoices: {df_country["InvoiceNo"].nunique()}')
print(f'Unique products: {df_country["Description"].nunique()}')

Transactions for France: 8019
Unique invoices: 379
Unique products: 1541


## 4. Build the "basket" matrix

We reshape the data so that:
- each **row** = one invoice (one shopping trip)
- each **column** = one product
- each **value** = how many units of that product were bought on that invoice (0 if not bought at all)

This is done with a pivot table.

In [22]:
basket = (
    df_country.groupby(['InvoiceNo', 'Description'])['Quantity']
    .sum()
    .unstack()          # turns the Description index level into columns
    .fillna(0)           # products not bought on an invoice become 0 instead of missing
)

print('Basket shape (invoices x products):', basket.shape)
basket.head()

Basket shape (invoices x products): (379, 1541)


Description   50'S CHRISTMAS GIFT BAG LARGE   DOLLY GIRL BEAKER  \
InvoiceNo                                                         
536370                                  0.0                 0.0   
536852                                  0.0                 0.0   
536974                                  0.0                 0.0   
537065                                  0.0                 0.0   
537463                                  0.0                 0.0   

Description   I LOVE LONDON MINI BACKPACK   NINE DRAWER OFFICE TIDY  \
InvoiceNo                                                             
536370                                0.0                       0.0   
536852                                0.0                       0.0   
536974                                0.0                       0.0   
537065                                0.0                       0.0   
537463                                0.0                       0.0   

Description   SET 2 TEA TOWELS I LOVE LONDON    SPACEBOY BABY GIFT SET  \
InvoiceNo                                                                
536370                                   24.0                      0.0   
536852                                    0.0                      0.0   
536974                                    0.0                      0.0   
537065                                    0.0                      0.0   
537463                                    0.0                      0.0   

Description   TRELLIS COAT RACK  10 COLOUR SPACEBOY PEN  \
InvoiceNo                                                 
536370                      0.0                     0.0   
536852                      0.0                     0.0   
536974                      0.0                     0.0   
537065                      0.0                     0.0   
537463                      0.0                     0.0   

Description  12 COLOURED PARTY BALLOONS  12 EGG HOUSE PAINTED WOOD  \
InvoiceNo                                                            
536370                              0.0                        0.0   
536852                              0.0                        0.0   
536974                              0.0                        0.0   
537065                              0.0                        0.0   
537463                              0.0                        0.0   

Description  12 MESSAGE CARDS WITH ENVELOPES  12 PENCIL SMALL TUBE WOODLAND  \
InvoiceNo                                                                     
536370                                   0.0                            0.0   
536852                                   0.0                            0.0   
536974                                   0.0                            0.0   
537065                                   0.0                            0.0   
537463                                   0.0                            0.0   

Description  12 PENCILS SMALL TUBE RED RETROSPOT  12 PENCILS SMALL TUBE SKULL  \
InvoiceNo                                                                       
536370                                       0.0                          0.0   
536852                                       0.0                          0.0   
536974                                       0.0                          0.0   
537065                                       0.0                          0.0   
537463                                       0.0                          0.0   

Description  12 PENCILS TALL TUBE POSY  12 PENCILS TALL TUBE RED RETROSPOT  \
InvoiceNo                                                                    
536370                             0.0                                 0.0   
536852                             0.0                                 0.0   
536974                             0.0                                 0.0   
537065                             0.0                                 0.0   
537463            

## 5. Convert to a "bought / not bought" table

The Apriori algorithm doesn't care *how many* units were bought — only *whether* a product was part of the order or not. So we convert every value to `True`/`False`: any quantity greater than 0 becomes `True` (bought), and 0 stays `False` (not bought).

We do this by comparing the whole table to 0 at once (`basket > 0`), which pandas applies element-by-element automatically — this is simpler and faster than looping through cells with `applymap` (which is also deprecated in recent pandas versions).

In [23]:
basket_encoded = basket > 0

basket_encoded.head()

Description   50'S CHRISTMAS GIFT BAG LARGE   DOLLY GIRL BEAKER  \
InvoiceNo                                                         
536370                                False               False   
536852                                False               False   
536974                                False               False   
537065                                False               False   
537463                                False               False   

Description   I LOVE LONDON MINI BACKPACK   NINE DRAWER OFFICE TIDY  \
InvoiceNo                                                             
536370                              False                     False   
536852                              False                     False   
536974                              False                     False   
537065                              False                     False   
537463                              False                     False   

Description   SET 2 TEA TOWELS I LOVE LONDON    SPACEBOY BABY GIFT SET  \
InvoiceNo                                                                
536370                                   True                    False   
536852                                  False                    False   
536974                                  False                    False   
537065                                  False                    False   
537463                                  False                    False   

Description   TRELLIS COAT RACK  10 COLOUR SPACEBOY PEN  \
InvoiceNo                                                 
536370                    False                   False   
536852                    False                   False   
536974                    False                   False   
537065                    False                   False   
537463                    False                   False   

Description  12 COLOURED PARTY BALLOONS  12 EGG HOUSE PAINTED WOOD  \
InvoiceNo                                                            
536370                            False                      False   
536852                            False                      False   
536974                            False                      False   
537065                            False                      False   
537463                            False                      False   

Description  12 MESSAGE CARDS WITH ENVELOPES  12 PENCIL SMALL TUBE WOODLAND  \
InvoiceNo                                                                     
536370                                 False                          False   
536852                                 False                          False   
536974                                 False                          False   
537065                                 False                          False   
537463                                 False                          False   

Description  12 PENCILS SMALL TUBE RED RETROSPOT  12 PENCILS SMALL TUBE SKULL  \
InvoiceNo                                                                       
536370                                     False                        False   
536852                                     False                        False   
536974                                     False                        False   
537065                                     False                        False   
537463                                     False                        False   

Description  12 PENCILS TALL TUBE POSY  12 PENCILS TALL TUBE RED RETROSPOT  \
InvoiceNo                                                                    
536370                           False                               False   
536852                           False                               False   
536974                           False                               False   
537065                           False                               False   
537463            

## 6. Find frequent itemsets with Apriori

The Apriori algorithm scans the basket table and finds combinations of products that appear together often enough to be meaningful. "Often enough" is controlled by `min_support`:

**Support** = (number of invoices containing this itemset) / (total number of invoices)

For example, `support = 0.05` means the itemset appears in at least 5% of all invoices. We start with a fairly low threshold (there are many products and each individual one is rare) and can raise it later if we get too many results to review.

In [24]:
frequent_itemsets = apriori(basket_encoded, min_support=0.05, use_colnames=True)

# Add a column with the number of products in each itemset, useful for filtering later
frequent_itemsets['itemset_length'] = frequent_itemsets['itemsets'].apply(len)

print(f'Found {len(frequent_itemsets)} frequent itemsets')
frequent_itemsets.sort_values('support', ascending=False).head(10)

Found 127 frequent itemsets


,support,itemsets,itemset_length
49,0.192612,frozenset({RABBIT NIGHT LIGHT}),1
57,0.184697,frozenset({RED TOADSTOOL LED NIGHT LIGHT}),1
47,0.176781,frozenset({PLASTERS IN TIN WOODLAND ANIMALS}),1
43,0.174142,frozenset({PLASTERS IN TIN CIRCUS PARADE }),1
64,0.163588,frozenset({ROUND SNACK BOXES SET OF4 WOODLAND }),1
27,0.158311,frozenset({LUNCH BAG RED RETROSPOT}),1
33,0.147757,frozenset({LUNCH BOX WITH CUTLERY RETROSPOT }),1
71,0.142480,frozenset({SET/6 RED SPOTTY PAPER CUPS}),1
55,0.142480,frozenset({RED RETROSPOT MINI CASES}),1
45,0.139842,frozenset({PLASTERS IN TIN SPACEBOY}),1


## 7. Generate association rules

A frequent itemset just says "these products often appear together" — it doesn't say which one predicts the other. **Association rules** turn itemsets into "if a customer buys A, they are likely to also buy B" statements, with three key metrics:

- **Support**: how often A and B appear together, out of all invoices (as above)
- **Confidence**: out of the invoices that contain A, what fraction also contain B — i.e. `P(B | A)`
- **Lift**: how much more likely B is to be bought when A is bought, compared to B's normal popularity on its own. `lift = 1` means no relationship, `lift > 1` means a positive relationship (they're bought together more than random chance would suggest), `lift < 1` means a negative relationship

In [25]:
rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1.0)

# Sort by lift descending to see the strongest relationships first
rules = rules.sort_values('lift', ascending=False)

print(f'Found {len(rules)} rules')
rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10)

Found 100 rules


,antecedents,consequents,support,confidence,lift
49,frozenset({PACK OF 6 SKULL PAPER PLATES}),frozenset({PACK OF 6 SKULL PAPER CUPS}),0.052770,0.909091,13.781818
48,frozenset({PACK OF 6 SKULL PAPER CUPS}),frozenset({PACK OF 6 SKULL PAPER PLATES}),0.052770,0.800000,13.781818
14,frozenset({CHILDRENS CUTLERY DOLLY GIRL }),frozenset({CHILDRENS CUTLERY SPACEBOY }),0.065963,0.892857,12.533069
15,frozenset({CHILDRENS CUTLERY SPACEBOY }),frozenset({CHILDRENS CUTLERY DOLLY GIRL }),0.065963,0.925926,12.533069
82,"frozenset({ALARM CLOCK BAKELIKE PINK, ALARM CL...",frozenset({ALARM CLOCK BAKELIKE RED }),0.065963,0.862069,8.830382
87,frozenset({ALARM CLOCK BAKELIKE RED }),"frozenset({ALARM CLOCK BAKELIKE PINK, ALARM CL...",0.065963,0.675676,8.830382
85,frozenset({ALARM CLOCK BAKELIKE GREEN}),"frozenset({ALARM CLOCK BAKELIKE PINK, ALARM CL...",0.065963,0.657895,8.598004
84,"frozenset({ALARM CLOCK BAKELIKE PINK, ALARM CL...",frozenset({ALARM CLOCK BAKELIKE GREEN}),0.065963,0.862069,8.598004
3,frozenset({ALARM CLOCK BAKELIKE RED }),frozenset({ALARM CLOCK BAKELIKE GREEN}),0.081794,0.837838,8.356330
2,frozenset({ALARM CLOCK BAKELIKE GREEN}),frozenset({ALARM CLOCK BAKELIKE RED }),0.081794,0.815789,8.356330


## 8. Interpret the top rules

`antecedents` = the product(s) already in the basket (the "if"). `consequents` = the product(s) likely to be added (the "then").

Read a row like: *antecedents = {Product A}, consequents = {Product B}, confidence = 0.75, lift = 6.2* as:

> "75% of customers who buy Product A also buy Product B — and they're about 6.2x more likely to do so than a random customer."

This is exactly the kind of insight that powers "frequently bought together" recommendations and cross-selling decisions.

In [26]:
# Filter to rules with strong confidence AND strong lift, for the clearest, most actionable relationships
strong_rules = rules[(rules['confidence'] >= 0.5) & (rules['lift'] >= 3)]

print(f'Strong rules (confidence >= 0.5 and lift >= 3): {len(strong_rules)}')
strong_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(15)

Strong rules (confidence >= 0.5 and lift >= 3): 53


,antecedents,consequents,support,confidence,lift
49,frozenset({PACK OF 6 SKULL PAPER PLATES}),frozenset({PACK OF 6 SKULL PAPER CUPS}),0.052770,0.909091,13.781818
48,frozenset({PACK OF 6 SKULL PAPER CUPS}),frozenset({PACK OF 6 SKULL PAPER PLATES}),0.052770,0.800000,13.781818
14,frozenset({CHILDRENS CUTLERY DOLLY GIRL }),frozenset({CHILDRENS CUTLERY SPACEBOY }),0.065963,0.892857,12.533069
15,frozenset({CHILDRENS CUTLERY SPACEBOY }),frozenset({CHILDRENS CUTLERY DOLLY GIRL }),0.065963,0.925926,12.533069
82,"frozenset({ALARM CLOCK BAKELIKE PINK, ALARM CL...",frozenset({ALARM CLOCK BAKELIKE RED }),0.065963,0.862069,8.830382
87,frozenset({ALARM CLOCK BAKELIKE RED }),"frozenset({ALARM CLOCK BAKELIKE PINK, ALARM CL...",0.065963,0.675676,8.830382
85,frozenset({ALARM CLOCK BAKELIKE GREEN}),"frozenset({ALARM CLOCK BAKELIKE PINK, ALARM CL...",0.065963,0.657895,8.598004
84,"frozenset({ALARM CLOCK BAKELIKE PINK, ALARM CL...",frozenset({ALARM CLOCK BAKELIKE GREEN}),0.065963,0.862069,8.598004
3,frozenset({ALARM CLOCK BAKELIKE RED }),frozenset({ALARM CLOCK BAKELIKE GREEN}),0.081794,0.837838,8.356330
2,frozenset({ALARM CLOCK BAKELIKE GREEN}),frozenset({ALARM CLOCK BAKELIKE RED }),0.081794,0.815789,8.356330


## 9. Save the rules

In [27]:
OUTPUT_PATH = Path('..') / 'data' / 'processed' / f'association_rules_{COUNTRY.lower()}.csv'

# frozenset objects (antecedents/consequents) need to be converted to plain text before saving to CSV
rules_to_save = rules.copy()
rules_to_save['antecedents'] = rules_to_save['antecedents'].apply(lambda x: ', '.join(sorted(x)))
rules_to_save['consequents'] = rules_to_save['consequents'].apply(lambda x: ', '.join(sorted(x)))

rules_to_save.to_csv(OUTPUT_PATH, index=False)
print(f'Saved {len(rules_to_save)} rules to: {OUTPUT_PATH.resolve()}')

Saved 100 rules to: D:\New folder\BasketIQ\data\processed\association_rules_france.csv


## 10. Optional: try another country

Once this works, you can go back to Section 3, change `COUNTRY` to another market with enough transactions (e.g. `'Germany'`), and re-run the notebook to compare shopping patterns across markets.